In [0]:
df = spark.table('gizmobox.bronze.refunds')
display(df)

1.Extract specific portion of the string from refund_reason using split function

[Documentation](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.regexp_extract.html)


In [0]:
from pyspark.sql.functions import split

df_extracted_refund = (
    df
    .select( 'refund_id',
        'payment_id',
        'refund_timestamp',
        'refund_amount',
        split('refund_reason',':')[0].alias('refund_reason'),
        split('refund_reason',':')[1].alias('refund_source') )
    
)

display(df_extracted_refund)



2.Extract specific portion of the string from refund_reason using regexp_extract function

[Documentation](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.split.html?highlight=split#pyspark.sql.functions.split)

In [0]:
from pyspark.sql.functions import regexp_extract

df_extracted_refund = (
    df
    .select( 'refund_id',
        'payment_id',
        'refund_timestamp',
        'refund_amount',
        regexp_extract('refund_reason','^([^:]+):',1).alias('refund_reason'),
        regexp_extract('refund_reason','^[^:]+:(.*)$',1).alias('refund_source') )
    
)

display(df_extracted_refund)

3.Extract date and time from refund_timestamp

In [0]:
from pyspark.sql.functions import regexp_extract,date_format

df_extracted_refund = (
    df
    .select( 'refund_id',
        'payment_id',
        date_format('refund_timestamp','yyyy-MM-dd').cast('date').alias('refund_date'),
        date_format('refund_timestamp','HH:mm:ss').alias('refund_time'),
        'refund_amount',
        regexp_extract('refund_reason','^([^:]+):',1).alias('refund_reason'),
        regexp_extract('refund_reason','^[^:]+:(.*)$',1).alias('refund_source') )
    
)

display(df_extracted_refund)


4. Write transformed data to the Silver schema

In [0]:
df_extracted_refund.writeTo('gizmobox.bronze.py_refunds').createOrReplace()

In [0]:
%sql
SELECT * FROM gizmobox.bronze.py_refunds;